In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 자신의 프로젝트 폴더 경로로 변경하세요
my_path = '/content/drive/MyDrive/비트메이트_TP01'

In [3]:
# =========================
# 1) 설치 + 모델 로드
# =========================

!pip install -q -U faster-whisper transformers accelerate bitsandbytes sentencepiece qwen-tts soundfile gradio numpy huggingface_hub hf_xet
!apt-get -qq update
!apt-get -qq install -y ffmpeg sox libsox-fmt-all


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 73.7 MB/s eta 0:00:00
   

In [4]:
# =========================
# 1) 모델 로드
# =========================
import os
import gc
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from faster_whisper import WhisperModel
from qwen_tts import Qwen3TTSModel
from huggingface_hub import login

my_path = Path('/content/drive/MyDrive/비트메이트_TP01')

# -------------------------
# Hugging Face 인증
# -------------------------
# dnotitia/DNA-2.0-14B는 공개 Hugging Face 모델입니다.
# 일반적으로 HF_TOKEN 없이도 로드할 수 있지만, 다운로드 제한/비공개 환경에 대비해
# Colab 왼쪽 열쇠 아이콘(Secrets)에 HF_TOKEN을 저장해 두면 더 안정적입니다.
#
# from huggingface_hub import notebook_login
# notebook_login()

HF_TOKEN = os.environ.get("HF_TOKEN")

# Colab Secrets에 HF_TOKEN을 넣은 경우 자동 사용
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("HF_TOKEN이 없습니다. 공개 모델이면 정상이며, 다운로드 제한이 걸리면 HF_TOKEN을 추가하세요.")

# -------------------------
# 빠른 로드를 위한 캐시 / 지연 로딩 설정
# -------------------------
HF_CACHE_DIR = Path("/content/hf_cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")

stt_device = "cuda" if torch.cuda.is_available() else "cpu"
stt_compute_type = "float16" if torch.cuda.is_available() else "int8"

# LLM: Qwen2.5-7B-Instruct -> DNA-2.0-14B
MODEL_NAME = "dnotitia/DNA-2.0-14B"

# 14B급 모델이라 Colab에서는 4bit 양자화를 기본값으로 둡니다.
# T4 16GB에서는 STT/TTS까지 함께 쓰면 메모리가 부족할 수 있습니다.
# A100 40GB/80GB처럼 여유가 있으면 quantization_config를 제거하고 torch_dtype=torch.bfloat16만 사용할 수 있습니다.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

speaker = "jjy"
ckpt_root = my_path / "qwen3_ft_output" / speaker

if not ckpt_root.exists():
    raise FileNotFoundError(f"경로를 찾을 수 없습니다: {ckpt_root}")

epochs = []
for name in os.listdir(ckpt_root):
    if name.startswith("checkpoint-epoch-"):
        try:
            epochs.append(int(name.split("-")[-1]))
        except ValueError:
            pass

if not epochs:
    raise RuntimeError(f"checkpoint-epoch-* 폴더를 찾지 못했습니다: {ckpt_root}")

# 체크포인트 선택
# selected_epoch = max(epochs)
selected_epoch = 1
target_ckpt = ckpt_root / f"checkpoint-epoch-{selected_epoch}"

whisper_model = None
tokenizer = None
llm_model = None
tts_model = None

def get_whisper_model():
    global whisper_model
    if whisper_model is None:
        whisper_model = WhisperModel(
            "large-v3-turbo",
            device=stt_device,
            compute_type=stt_compute_type,
            download_root=str(HF_CACHE_DIR / "faster_whisper"),
        )
        print(
            "Whisper(faster-whisper) 로드 완료 | "
            f"language=ko 고정 | device={stt_device} | compute_type={stt_compute_type}"
        )
    return whisper_model

def _first_model_device(model):
    """
    device_map='auto' 환경에서 입력 텐서를 올릴 장치를 안전하게 찾습니다.
    """
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_llm_components():
    global tokenizer, llm_model
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            cache_dir=str(HF_CACHE_DIR / "transformers"),
            local_files_only=False,
            token=HF_TOKEN,
            trust_remote_code=False,
        )

    if llm_model is None:
        llm_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            cache_dir=str(HF_CACHE_DIR / "transformers"),
            local_files_only=False,
            token=HF_TOKEN,
            trust_remote_code=False,
        )
        llm_model.eval()
        print("DNA-2.0-14B 로드 완료")

    return tokenizer, llm_model

def get_tts_model():
    global tts_model
    if tts_model is None:
        tts_model = Qwen3TTSModel.from_pretrained(
            str(target_ckpt),
            device_map="cuda:0" if torch.cuda.is_available() else "cpu",
            dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        print("TTS 로드 완료")
    return tts_model

def preload_all_models():
    print("🚀 모델 전체 사전 로딩 시작...")

    # 1. STT
    print("👉 Whisper 로딩 중...")
    _ = get_whisper_model()

    # 2. LLM
    print("👉 LLM 로딩 중...")
    _ = get_llm_components()

    # 3. TTS
    print("👉 TTS 로딩 중...")
    _ = get_tts_model()

    torch.cuda.empty_cache()
    gc.collect()

    print("✅ 모든 모델 로딩 완료 (STT + LLM + TTS)")

def preprocess_audio_for_stt(audio_path, target_sr=16000):
    audio_path = Path(audio_path)
    audio, sr = sf.read(str(audio_path))

    # stereo -> mono
    if getattr(audio, "ndim", 1) > 1:
        audio = audio.mean(axis=1)

    audio = audio.astype(np.float32)

    # normalize volume safely
    peak = np.max(np.abs(audio)) if len(audio) > 0 else 0.0
    if peak > 0:
        audio = audio / peak * 0.95

    # simple linear resample to 16k if needed
    if sr != target_sr and len(audio) > 0:
        duration = len(audio) / sr
        old_t = np.linspace(0, duration, num=len(audio), endpoint=False)
        new_len = int(duration * target_sr)
        new_t = np.linspace(0, duration, num=new_len, endpoint=False)
        audio = np.interp(new_t, old_t, audio).astype(np.float32)
        sr = target_sr

    out_path = audio_path.with_name(audio_path.stem + "_stt_ko.wav")
    sf.write(str(out_path), audio, sr)
    return out_path

def clean_stt_text(text):
    text = text.strip()

    bad_prefixes = [
        "다음은 한국어 음성입니다.",
        "한국어 음성입니다.",
        "한국어로 정확하게 받아쓰기 하세요.",
        "다음은 한국어입니다.",
    ]
    for prefix in bad_prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    return text

def looks_like_hallucination(text):
    t = text.strip()

    if not t:
        return True

    digit_count = sum(ch.isdigit() for ch in t)
    digit_ratio = digit_count / max(len(t), 1)

    if digit_ratio > 0.45:
        return True

    if len(t) >= 6 and len(set(t)) <= 3:
        return True

    return False

def transcribe_audio(audio_path):
    model = get_whisper_model()
    prepared_audio_path = preprocess_audio_for_stt(audio_path)

    segments, info = model.transcribe(
        str(prepared_audio_path),
        language="ko",
        task="transcribe",
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=500,
            speech_pad_ms=200,
        ),
        beam_size=5,
        best_of=5,
        temperature=0.0,
        condition_on_previous_text=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        compression_ratio_threshold=2.0,
        log_prob_threshold=-0.8,
        no_speech_threshold=0.75,
    )

    text = "".join(segment.text for segment in segments).strip()
    text = clean_stt_text(text)

    print(
        "STT 디버그 | "
        f"detected_language={getattr(info, 'language', 'unknown')} | "
        f"language_probability={getattr(info, 'language_probability', 'unknown')}"
    )

    if looks_like_hallucination(text):
        print("STT 경고 | 환각 가능성이 높은 결과라서 빈 문자열로 처리")
        return ""

    return text

preload_all_models()
print("- LLM은 dnotitia/DNA-2.0-14B로 설정되었습니다.")
print("- DNA-2.0-14B는 공개 모델입니다. 다운로드 제한이 있으면 HF_TOKEN을 사용하세요.")
print("- STT는 한국어 고정 + 전처리(모노/정규화/16kHz) + VAD + 환각 억제 설정이 적용됩니다.")



********
********
 
HF_TOKEN이 없습니다. 공개 모델이면 정상이며, 다운로드 제한이 걸리면 HF_TOKEN을 추가하세요.
🚀 모델 전체 사전 로딩 시작...
👉 Whisper 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00006-of-00006.safetensors:   0%|          | 0.00/4.73G [00:00<?, ?B/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

DNA-2.0-14B 로드 완료
👉 TTS 로딩 중...
TTS 로드 완료
✅ 모든 모델 로딩 완료 (STT + LLM + TTS)
- LLM은 dnotitia/DNA-2.0-14B로 설정되었습니다.
- DNA-2.0-14B는 공개 모델입니다. 다운로드 제한이 있으면 HF_TOKEN을 사용하세요.
- STT는 한국어 고정 + 전처리(모노/정규화/16kHz) + VAD + 환각 억제 설정이 적용됩니다.


In [5]:
# =========================
# 2) 함수
# =========================
import re


def contains_chinese(text, threshold=2):
    """
    응답에 중국어 한자가 일정 수 이상 포함되는지 확인
    """
    count = 0
    for ch in text:
        code = ord(ch)
        if (
            0x4E00 <= code <= 0x9FFF or   # CJK Unified Ideographs
            0x3400 <= code <= 0x4DBF or   # CJK Extension A
            0xF900 <= code <= 0xFAFF      # CJK Compatibility Ideographs
        ):
            count += 1
    return count >= threshold



def strip_thinking_text(text):
    """
    Qwen3 계열 모델이 <think>...</think> 형태의 사고 과정을 출력하는 경우,
    TTS로 읽히지 않도록 최종 답변만 남깁니다.
    """
    text = text.strip()

    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = text.replace("<think>", "").replace("</think>", "").strip()
    return text


def _generate_llm_response(messages, max_new_tokens=256):
    tokenizer, llm_model = get_llm_components()

    # DNA-2.0-14B는 Qwen3 계열이라 tokenizer가 지원하면 thinking 출력을 끕니다.
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    input_device = _first_model_device(llm_model)
    inputs = tokenizer(text, return_tensors="pt").to(input_device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id if tokenizer.pad_token_id is None else tokenizer.pad_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    response = strip_thinking_text(response)
    return response


def chat_llm(user_text, system_prompt=None):
    persona_prompt = system_prompt or "당신은 AI 비서이다."

    base_prompt = (
        "모든 답변은 반드시 자연스러운 한국어로만 작성하라. "
        "중국어, 한자, 영어로 답하지 마라. "
        "항상 부드럽고 명확한 한국어 문장으로 답하라. "
        "사고 과정이나 <think> 태그를 출력하지 마라. "
        "괄호를 사용해서 부연 설명하지 마라."
    )

    final_system_prompt = f"{persona_prompt}\n{base_prompt}"

    messages = [
        {"role": "system", "content": final_system_prompt},
        {"role": "assistant", "content": "알겠다. 앞으로 모든 답변은 한국어로만 하겠다."},
        {"role": "user", "content": user_text},
    ]

    response = _generate_llm_response(messages)

    if contains_chinese(response):
        retry_messages = [
            {
                "role": "system",
                "content": (
                    f"{persona_prompt}\n"
                    "반드시 한국어로만 답하라. "
                    "중국어, 한자, 영어를 절대 사용하지 마라. "
                    "자연스러운 한국어 문장만 출력하라. "
                    "사고 과정이나 <think> 태그를 출력하지 마라. "
                    "괄호를 사용해서 부연 설명하지 마라."
                )
            },
            {"role": "assistant", "content": "알겠다. 반드시 한국어로만 답하겠다."},
            {"role": "user", "content": user_text},
        ]
        response = _generate_llm_response(retry_messages)

    return response


def record_audio(sec=10, filename='/content/recorded_audio.wav'):
  print(f"🎤 {sec}초 동안 녹음을 시작합니다...")

  # JS 실행
  display(Javascript(RECORD_JS))

  # JS 함수 호출 및 데이터 수신
  s = output.eval_js(f'record({sec * 1000})')

  # Base64 디코딩
  b = b64decode(s.split(',')[1])

  # 파일 저장
  with open(filename, 'wb') as f:
    f.write(b)

  print(f"✅ 녹음 완료! 파일이 '{filename}'로 저장되었습니다.")


In [6]:
import gradio as gr
import soundfile as sf

def run_pipeline(input_mode, audio_path, text_input, manual_tts_text, persona_prompt):
    # 1) LLM 답변 칸에 직접 입력한 텍스트가 있으면 바로 TTS
    if manual_tts_text and manual_tts_text.strip():
        question = text_input.strip() if text_input else ""
        answer = manual_tts_text.strip()

    else:
        # 2) 입력 처리
        if input_mode == "음성 입력":
            question = transcribe_audio(audio_path)
        else:
            question = text_input.strip()

        if not question:
            return "입력을 확인해주세요.", "입력을 확인해주세요.", None

        # 3) LLM
        answer = chat_llm(question, system_prompt=persona_prompt)

    # 4) TTS
    tts_model = get_tts_model()
    wavs, sr = tts_model.generate_custom_voice(
        text=answer,
        speaker=speaker
    )

    out_path = "/content/gradio_reply.wav"
    sf.write(out_path, wavs[0], sr)

    return question, answer, out_path


def toggle_input(mode):
    return (
        gr.update(visible=(mode == "음성 입력")),
        gr.update(visible=(mode == "텍스트 입력"))
    )


with gr.Blocks() as demo:
    gr.Markdown("## 🎤 음성 / 텍스트 대화 테스트")

    input_mode = gr.Radio(
        choices=["음성 입력", "텍스트 입력"],
        value="음성 입력",
        label="입력 방식 선택"
    )

    audio_in = gr.Audio(
        sources=["microphone"],
        type="filepath",
        format="wav",
        label="마이크 입력",
        visible=True
    )

    text_input = gr.Textbox(
        label="텍스트 입력",
        placeholder="여기에 입력하세요",
        visible=False
    )

    input_mode.change(
        fn=toggle_input,
        inputs=input_mode,
        outputs=[audio_in, text_input]
    )

    persona_input = gr.Textbox(
        label="AI 페르소나 설정",
        value="당신은 자녀에게 대답하는 자상한 아버지이다."
    )

    run_btn = gr.Button("실행")

    stt_box = gr.Textbox(label="입력 내용")

    llm_box = gr.Textbox(
        label="LLM 답변 / 직접 TTS 입력",
        placeholder="여기에 직접 문장을 넣고 실행하면 LLM 없이 바로 TTS로 변환됩니다.",
        interactive=True
    )

    tts_audio = gr.Audio(label="TTS 출력", type="filepath")

    run_btn.click(
        fn=run_pipeline,
        inputs=[input_mode, audio_in, text_input, llm_box, persona_input],
        outputs=[stt_box, llm_box, tts_audio]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f82a1825e7efe6fe4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
